# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os
if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.chdir("/content/rdt-igtesting")
os.system("git pull -q")

TASKS = ["PickCube-v1", "StackCube-v1", "PushCube-v1", "PegInsertionSide-v1", "PlugCharger-v1"]
for task in TASKS:
    shutil.copy(
        hf_hub_download("robotics-diffusion-transformer/maniskill-model",
                        "lang_embeds/text_embed_" + task + ".pt"),
        "lang_embed_" + task + ".pt"
    )
    print("Downloaded lang embed:", task)

shutil.copy("lang_embed_PickCube-v1.pt", "lang_embed.pt")
print("Ready.")


In [ ]:
import subprocess, sys, random
base_seed = random.randint(0, 99999)
print("Using base_seed:", base_seed)
proc = subprocess.Popen(
    [sys.executable, "rollout_subprocess.py",
     "--task", "PickCube-v1", "--n", "25", "--base-seed", str(base_seed)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd="/content/rdt-igtesting"
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("Subprocess exited: code", proc.returncode)


In [ ]:
%matplotlib inline
import json, os
import torch, numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image as IPyImage

with open('success_frames.json') as f:
    meta = json.load(f)
task           = meta['task']
success_frames = meta['successes']
print('Task:', task, ' | ', len(success_frames), 'successful episodes')
if not success_frames:
    print('No successes to analyse.'); raise SystemExit

import os as _os
_os.environ['MANISKILL_TASK'] = task
_os.environ['LANG_EMBED']     = 'lang_embed_' + task + '.pt'
_os.environ['SKIP_IG']        = '1'
if 'rdt' not in dir():
    %run rdt_blurig.py

from transformers import AutoTokenizer
_tok  = AutoTokenizer.from_pretrained('t5-small')
L_emb = lang_tokens.shape[1]
task2lang = {
    'PickCube-v1':         'Grasp a red cube and move it to a target goal position.',
    'StackCube-v1':        'Pick up a red cube and stack it on top of a green cube.',
    'PushCube-v1':         'Push and move a cube to a goal region in front of it.',
    'PegInsertionSide-v1': 'Pick up a orange-white peg and insert the orange end into the box.',
    'PlugCharger-v1':      'Pick up one of the misplaced shapes and insert it into the correct slot.',
}
_desc = task2lang.get(task, task)
_ids  = _tok(_desc, return_tensors='pt').input_ids[0]
n_use = min(len(_ids), L_emb)
raw_words = _tok.convert_ids_to_tokens(_ids[:n_use])
groups, cur = [], [0]
for i in range(1, n_use):
    if raw_words[i].startswith(chr(9601)): groups.append(cur); cur = [i]
    else: cur.append(i)
groups.append(cur)
for i in range(n_use, L_emb): groups.append([i])
plot_labels = [
    (_tok.decode([_ids[i].item() for i in g if i < len(_ids)], skip_special_tokens=True).strip() or '[' + str(g[0]) + ']')
    if g[0] < n_use else '[pad]' for g in groups
]
W = len(plot_labels)
JOINT_NAMES = ['base rot','shoulder','upper arm','elbow','forearm rot','wrist pitch','wrist rot','gripper']
_midx = torch.tensor(MANISKILL_INDICES, device=DEVICE)

N_IG = 15
_bl  = torch.zeros_like(lang_tokens)
_dl  = lang_tokens - _bl

def _blur_emb(e, sigma):
    import torch.nn.functional as F
    B,T,C = e.shape; G = int(T**0.5)
    m = e.reshape(B,G,G,C).permute(0,3,1,2)
    if sigma > 0:
        ks = max(3, int(6*sigma)//2*2+1)
        m = F.avg_pool2d(m, kernel_size=ks, stride=1, padding=ks//2)
    return m.permute(0,2,3,1).reshape(B,T,C)

def run_ig(scene_emb, scene_state, am0):
    wj = [torch.zeros_like(lang_tokens) for _ in range(8)]
    for k in range(N_IG):
        alpha  = (k + 0.5) / N_IG
        interp = (_bl + alpha * _dl).requires_grad_(True)
        with torch.enable_grad():
            _ac = rdt.conditional_sample(
                rdt.lang_adaptor(interp), lang_attn_mask,
                rdt.img_adaptor(scene_emb.detach().repeat(1,6,1)),
                scene_state, am0, ctrl_freqs)
        js = _ac[:, :8, :][:, :, _midx].norm(dim=(0,1))
        for j in range(8):
            g = torch.autograd.grad(js[j], interp, retain_graph=(j<7))[0]
            wj[j] = wj[j] + g.detach() * _dl.detach()
    attr_jt = np.array([wj[j].squeeze(0).float().cpu().abs().sum(-1).numpy() for j in range(8)])
    attr_jw = np.array([[attr_jt[j, g].sum() for g in groups] for j in range(8)])
    sigmas  = [10.0, 7.5, 5.0, 2.5, 0.0]
    wi_tot  = torch.zeros_like(scene_emb)
    for k in range(len(sigmas)-1):
        E_t    = _blur_emb(scene_emb.detach(), sigmas[k]).requires_grad_(True)
        E_next = _blur_emb(scene_emb.detach(), sigmas[k+1])
        with torch.enable_grad():
            _ac = rdt.conditional_sample(rdt.lang_adaptor(lang_tokens), lang_attn_mask,
                                         rdt.img_adaptor(E_t.repeat(1,6,1)),
                                         scene_state, am0, ctrl_freqs)
        score = _ac[:, :8, :][:, :, _midx].norm()
        g = torch.autograd.grad(score, E_t)[0]
        wi_tot = wi_tot + g.detach() * (E_next - E_t.detach())
    G      = int(scene_emb.shape[1]**0.5)
    wi_map = wi_tot.squeeze(0).float().cpu().abs().sum(-1).reshape(G, G).numpy()
    wi_map = (wi_map - wi_map.min()) / (wi_map.max() - wi_map.min() + 1e-8)
    return attr_jw, wi_map

_bil = getattr(PILImage, 'Resampling', PILImage).BILINEAR

for info in success_frames:
    ep    = info['ep']
    seed  = info['seed']
    steps = info['steps']
    flist = info['frames']
    print('Episode', ep, 'seed=', seed, 'steps=', steps)

    fig1, axes1 = plt.subplots(1, 10, figsize=(20, 2))
    for k, fd in enumerate(flist):
        img_k = PILImage.open(fd['path']).convert('RGB')
        axes1[k].imshow(np.array(img_k))
        label = 'start' if k == 0 else ('SUCCESS' if k == 9 else 'step ' + str(fd['step']))
        axes1[k].set_title(label, fontsize=6)
        axes1[k].axis('off')
    fig1.suptitle(task + '  ep' + str(ep) + '  seed' + str(seed) + '  (' + str(steps) + ' steps)', fontsize=9)
    plt.tight_layout()
    strip_path = 'strip_ep' + str(ep).zfill(2) + '.png'
    plt.savefig(strip_path, dpi=120, bbox_inches='tight')
    plt.close()
    display(IPyImage(strip_path))

    scene_frame = PILImage.open(flist[0]['path']).convert('RGB')
    with torch.no_grad():
        scene_emb = encode_image(scene_frame)
        _s0 = torch.zeros(1,1,128,dtype=DTYPE,device=DEVICE)
        _a0 = torch.zeros(1,1,128,dtype=DTYPE,device=DEVICE)
        _a0[0,0,MANISKILL_INDICES] = 1.0
        scene_state = rdt.state_adaptor(torch.cat([_s0,_a0],dim=2))

    print('  Running IG ...')
    attr_jw, wi_map = run_ig(scene_emb, scene_state, _a0)
    img_np = np.array(scene_frame) / 255.0
    up_wi  = np.array(PILImage.fromarray((wi_map*255).astype(np.uint8)).resize((384,384),_bil)) / 255.0

    fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4))
    axes2[0].imshow(img_np); axes2[0].set_title('Start frame'); axes2[0].axis('off')
    axes2[1].imshow(img_np)
    axes2[1].imshow(up_wi, cmap='inferno', alpha=0.6, vmin=0, vmax=1)
    axes2[1].set_title('Word x Image (BlurIG)'); axes2[1].axis('off')
    _jw_n = attr_jw / (attr_jw.max(axis=1, keepdims=True) + 1e-8)
    im = axes2[2].imshow(_jw_n, cmap='inferno', aspect='auto', vmin=0, vmax=1)
    axes2[2].set_xticks(range(W)); axes2[2].set_xticklabels(plot_labels, rotation=40, ha='right', fontsize=8)
    axes2[2].set_yticks(range(8)); axes2[2].set_yticklabels(JOINT_NAMES, fontsize=8)
    axes2[2].set_title('Word x Joint (token IG)')
    plt.colorbar(im, ax=axes2[2], fraction=0.03)
    plt.suptitle(task + '  ep' + str(ep) + '  seed' + str(seed), fontsize=10)
    plt.tight_layout()
    ig_path = 'ig_ep' + str(ep).zfill(2) + '.png'
    plt.savefig(ig_path, dpi=120, bbox_inches='tight')
    plt.close()
    display(IPyImage(ig_path))
    print()

print('Done.')
